# Unit 9 -- Direct collocation

**4SC000 Optimal Control and Reinforcement Learning**

Collocation does *not* eliminate the states by simulation. States and
controls at every grid node are decision variables, and the dynamics appear
only as equality constraints ("defects") tying consecutive nodes together.
That trades a small dense problem for a much larger but sparse and
well-conditioned one.

Two schemes, both obtained by applying a quadrature rule to
$x(t+\tau)-x(t)=\int f\,dt$:

- **trapezoidal**, $\;x_{k+1}-x_k-\frac{\tau}{2}(f_k+f_{k+1})=0$, second order;
- **Hermite-Simpson**, which interpolates $x$ by a cubic Hermite polynomial,
  evaluates the dynamics at the interval midpoint of that interpolant and
  applies Simpson's rule -- fourth order.

The second half of this notebook does something the slides only assert:
it **checks whether the answers are trajectories the pendulum would actually
follow**.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.figsize": (10, 3.6), "axes.grid": True,
                     "grid.alpha": 0.3, "font.size": 11})

# the swing-up problem used throughout Unit 9
x0      = np.array([0.0, 0.0])      # hanging down, at rest
xtarget = np.array([np.pi, 0.0])    # inverted, at rest
T, h    = 6.0, 60                   # horizon and number of steps
tau     = T / h
t_grid  = np.arange(h + 1) * tau
print(f"h = {h} steps of tau = {tau:.3f} s over T = {T} s")

In [ ]:
import collocation, shooting

xs_tr, u_tr, eff_tr = collocation.trapezoidal_collocation_solve(x0, xtarget, h, tau)
xs_hs, u_hs, eff_hs = collocation.hermite_simpson_collocation_solve(x0, xtarget, h, tau)

for name, xs_, eff in (("trapezoidal", xs_tr, eff_tr),
                       ("Hermite-Simpson", xs_hs, eff_hs)):
    print(f"  {name:16s} theta(T) = {xs_[-1,0]:.4f}   effort = {eff:.4f}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
ax[0].plot(t_grid, xs_tr[:, 0], lw=2, label="trapezoidal")
ax[0].plot(t_grid, xs_hs[:, 0], lw=2, ls="--", label="Hermite-Simpson")
ax[0].axhline(np.pi, color="0.5", ls=":")
ax[0].set_xlabel("t"); ax[0].set_ylabel(r"$\theta(t)$"); ax[0].legend(fontsize=9)
ax[1].plot(t_grid, u_tr, lw=2); ax[1].plot(t_grid, u_hs, lw=2, ls="--")
ax[1].set_xlabel("t"); ax[1].set_ylabel("u(t)")
plt.tight_layout(); plt.show()

## Accuracy as the grid is refined -- or so it looks

Theory says the two schemes should separate as $N$ grows: fourth order buys
far more per node than second order. Run the sweep and look carefully at what
actually happens.

In [ ]:
Ns = [10, 20, 30, 40]
eff_t, eff_h = [], []
for N in Ns:
    tt = T / N
    _, _, a = collocation.trapezoidal_collocation_solve(x0, xtarget, N, tt)
    _, _, b = collocation.hermite_simpson_collocation_solve(x0, xtarget, N, tt)
    eff_t.append(a); eff_h.append(b)
    print(f"  N = {N:3d}   trapezoidal {a:.5f}   Hermite-Simpson {b:.5f}")

plt.figure(figsize=(6.4, 3.6))
plt.plot(Ns, eff_t, "o-", lw=1.8, label="trapezoidal")
plt.plot(Ns, eff_h, "s-", lw=1.8, label="Hermite-Simpson")
plt.xlabel("number of intervals $N$"); plt.ylabel(r"reported optimal $\int u^2 dt$")
plt.legend(); plt.tight_layout(); plt.show()

Two things in that plot should bother you.

The curve **never flattens** -- a converging quantity levels off, and this one
is still falling at the right-hand edge. And the two schemes are
**indistinguishable**, when one of them is supposed to be two orders more
accurate than the other.

Both symptoms have the same cause, and it is not the quadrature.

## Is the answer a trajectory the pendulum would follow?

Here is the question the reported numbers above do **not** answer. The
collocation solution is a set of node values that very nearly satisfies the
defects. "Very nearly" is doing work: `collocation.py` enforces the defects
as *penalized* least-squares residuals with a weight $\rho$, not as hard
constraints, so the optimizer is free to buy a lower effort by violating the
dynamics a little.

The honest test is to throw the node values away, keep only the control,
and simulate it through the true dynamics.

In [ ]:
def audit(u_nodes, kind, sub=50):
    """Apply a control to the TRUE dynamics on a fine grid and integrate the
    effort.  kind = 'zoh' (one value per step, as shooting and iLQR produce)
    or 'pwl' (values at the N+1 nodes, linearly interpolated, as collocation
    produces).  Returns (final state, effort)."""
    dt, nodes = tau / sub, np.arange(h + 1) * tau
    x, eff = x0.copy(), 0.0
    for i in range(h * sub):
        t = i * dt
        if kind == "zoh":
            u = u_next = u_nodes[min(int(t / tau), h - 1)]
        else:
            u, u_next = np.interp([t, t + dt], nodes, u_nodes)
        x = shooting.rk4_step(x, u, dt)
        eff += 0.5 * dt * (u**2 + u_next**2)
    return x, eff


def defect_norm(xs, u, tt):
    d = [xs[k+1] - xs[k] - (tt/2) * (collocation.f_cont(xs[k], u[k])
                                     + collocation.f_cont(xs[k+1], u[k+1]))
         for k in range(len(u) - 1)]
    return np.max(np.abs(np.array(d)))


x_end, eff_sim = audit(u_tr, "pwl")
print(f"trapezoidal, default rho = 300")
print(f"   max |defect|              {defect_norm(xs_tr, u_tr, tau):.2e}")
print(f"   effort the solver reports {eff_tr:.4f}")
print(f"   effort when simulated     {eff_sim:.4f}")
print(f"   theta(T) when simulated   {x_end[0]:.4f}   (target {np.pi:.4f})")

The control that was supposed to bring the pendulum to $\theta=\pi$ leaves
it less than halfway there. The reported effort of about $2.03$ is not the
cost of a swing-up at all -- it is the cost of a trajectory that does not
swing up, bought by letting the defects drift.

Raising the penalty weight fixes it.

In [ ]:
for rho in (300.0, 3000.0):
    xs_r, u_r, eff_r = collocation.trapezoidal_collocation_solve(
        x0, xtarget, h, tau, rho=rho)
    x_end, eff_sim = audit(u_r, "pwl")
    print(f"  rho = {rho:7.0f}   max|defect| = {defect_norm(xs_r, u_r, tau):.2e}   "
          f"reported {eff_r:.4f}   simulated {eff_sim:.4f}   "
          f"theta(T) = {x_end[0]:.4f}")

Now redo the refinement sweep at the tighter weight. The defect violation
per interval is roughly constant at a given $\rho$, so the *more* intervals
the grid has, the more total drift the optimizer can buy -- which is exactly
why the first sweep fell forever instead of converging.

In [ ]:
eff_t3, eff_h3 = [], []
for N in Ns:
    tt = T / N
    _, _, a = collocation.trapezoidal_collocation_solve(x0, xtarget, N, tt, rho=3000.0)
    _, _, b = collocation.hermite_simpson_collocation_solve(x0, xtarget, N, tt, rho=3000.0)
    eff_t3.append(a); eff_h3.append(b)
    print(f"  N = {N:3d}   trapezoidal {a:.5f}   Hermite-Simpson {b:.5f}")

plt.figure(figsize=(6.4, 3.6))
plt.plot(Ns, eff_t, "o--", lw=1.4, color="0.6", label=r"trapezoidal, $\rho=300$")
plt.plot(Ns, eff_t3, "o-", lw=1.8, label=r"trapezoidal, $\rho=3000$")
plt.plot(Ns, eff_h3, "s-", lw=1.8, label=r"Hermite-Simpson, $\rho=3000$")
plt.xlabel("number of intervals $N$"); plt.ylabel(r"reported optimal $\int u^2 dt$")
plt.legend(fontsize=9); plt.tight_layout(); plt.show()

At $\rho=3000$ the sweep behaves like a convergence study: it settles
towards about $2.22$, which is where multiple shooting independently lands.

The two schemes are still almost on top of each other, and that is the real
lesson of this notebook. **In a penalty formulation it is the penalty weight,
not the order of the quadrature, that sets the accuracy.** Hermite-Simpson's
fourth order cannot show through an error floor imposed by $\rho$.

To see that order *does* matter when the dynamics are imposed as genuine
constraints, look at the energy-optimal case study in
`A_9_EVCaseStudies.ipynb`: there the defects reach an SQP solver as hard
equality constraints, and Hermite-Simpson at $N=20$ matches trapezoidal at
$N=80$, exactly as the theory promises.

## All five methods, scored the same way

With that in hand, every method of this unit can be compared honestly:
simulate its control, see where the pendulum ends up, and integrate the same
cost functional on the same fine grid.

In [ ]:
import ilqr

u1, _, _ = shooting.single_shooting_solve(x0, xtarget, h, tau)
u2, _, _, _ = shooting.multiple_shooting_solve(x0, xtarget, h, 6, tau)
_, u3, _ = collocation.trapezoidal_collocation_solve(x0, xtarget, h, tau, rho=3000.0)
_, u4, _ = collocation.hermite_simpson_collocation_solve(x0, xtarget, h, tau, rho=3000.0)
u5, _, _, _ = ilqr.ilqr_solve(x0, xtarget, h, tau)

print(f"  {'method':26s} {'theta(T)':>9s} {'miss':>8s} {'effort':>9s}")
for name, u_, kind in (("single shooting", u1, "zoh"),
                       ("multiple shooting", u2, "zoh"),
                       ("trapezoidal (rho=3000)", u3, "pwl"),
                       ("Hermite-Simpson (rho=3000)", u4, "pwl"),
                       ("iLQR", u5, "zoh")):
    xe, ef = audit(u_, kind)
    print(f"  {name:26s} {xe[0]:9.4f} {np.linalg.norm(xe-xtarget):8.4f} {ef:9.4f}")

Read the table rather than the headline numbers. The methods split into the
two local optima of the previous notebook, and within each group they agree
closely -- which is the right outcome: they are different discretizations of
the same problem, not different problems.

The lesson generalizes well beyond this pendulum, and it returns in the
closing case studies of the unit: **the objective value an optimizer reports
is computed on its own grid, by its own rule, subject to its own tolerances.
It is not the cost of the trajectory you will fly. Simulate it.**

Continue with `A_9_ILQRPendulum.ipynb`.